### Project Overview

HealthConnect Clinic is a fictional healthcare provider facing challenges with missed appointments, inefficient use of appointment slots, and limited insight into the factors driving no-shows. The clinic wants to explore how data and AI can support better decision-making and improve the patient experience.

As the Data Science track, my responsibility within this shared project is to define the machine learning problem and assess whether the available appointment data can support a no-show prediction solution.

Resources Used are
1. HealthConnect_Appointment_Data.csv — the appointment dataset.
2. HealthConnect_Data_Dictionary.xlsx — variable definitions and explanations used to interpret the dataset correctly (e.g. confirming waiting_time_minutes is an estimated which is a pre-appointment value)

The approach for Week 4 follows a structured problem-understanding process which is to inspect and assess the quality of the dataset, define the machine learning problem, propose a target variable, identify candidate input features, and outline an initial modelling approach  before any actual model building begins.

In [1]:
## importing libraries

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
health_connect_appointment = pd.read_csv("HealthConnect_Appointment_Data.csv")
health_connect_appointment.head()

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


In [4]:
health_connect_appointment.shape

(5000, 18)

In [5]:
health_connect_appointment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   appointment_id         5000 non-null   object 
 1   patient_id             5000 non-null   object 
 2   gender                 5000 non-null   object 
 3   age                    5000 non-null   int64  
 4   age_group              5000 non-null   object 
 5   appointment_type       5000 non-null   object 
 6   booking_date           5000 non-null   object 
 7   appointment_date       5000 non-null   object 
 8   appointment_day        5000 non-null   object 
 9   appointment_time       5000 non-null   object 
 10  booking_lead_days      5000 non-null   int64  
 11  previous_appointments  5000 non-null   int64  
 12  previous_no_shows      5000 non-null   int64  
 13  reminder_sent          5000 non-null   object 
 14  reminder_channel       3634 non-null   object 
 15  dist

In [6]:
health_connect_appointment.isnull().sum()

appointment_id              0
patient_id                  0
gender                      0
age                         0
age_group                   0
appointment_type            0
booking_date                0
appointment_date            0
appointment_day             0
appointment_time            0
booking_lead_days           0
previous_appointments       0
previous_no_shows           0
reminder_sent               0
reminder_channel         1366
distance_to_clinic_km      90
waiting_time_minutes       60
appointment_outcome         0
dtype: int64

In [7]:
## Verifying missingness logic for reminder_channel
mismatch = ((health_connect_appointment['reminder_sent'] == 'No') & 
            (health_connect_appointment['reminder_channel'].notnull())).sum()
mismatch2 = ((health_connect_appointment['reminder_sent'] == 'Yes') & 
             (health_connect_appointment['reminder_channel'].isnull())).sum()
print("Reminder sent=No but channel present:", mismatch)
print("Reminder sent=Yes but channel missing:", mismatch2)

Reminder sent=No but channel present: 0
Reminder sent=Yes but channel missing: 0


In [8]:
## Verifying previous_no_shows never exceeds previous_appointments
invalid = (health_connect_appointment['previous_no_shows'] > 
           health_connect_appointment['previous_appointments']).sum()
print("Rows where previous_no_shows > previous_appointments:", invalid)

Rows where previous_no_shows > previous_appointments: 0


In [9]:
health_connect_appointment.describe()


,age,booking_lead_days,previous_appointments,previous_no_shows,distance_to_clinic_km,waiting_time_minutes
count,5000.000000,5000.00000,5000.000000,5000.000000,4910.000000,4940.000000
mean,48.794800,29.63860,3.013800,0.544200,10.109572,24.189676
std,18.138547,17.39936,1.741211,0.746832,6.590030,10.863184
min,18.000000,0.00000,0.000000,0.000000,0.500000,2.000000
25%,33.000000,15.00000,2.000000,0.000000,5.300000,17.000000
50%,49.000000,30.00000,3.000000,0.000000,8.700000,24.000000
75%,64.000000,45.00000,4.000000,1.000000,13.500000,32.000000
max,80.000000,60.00000,11.000000,5.000000,45.000000,68.000000


In [10]:
categorical_cols = ['gender', 'appointment_type', 'reminder_sent', 'reminder_channel', 
                     'appointment_day', 'appointment_time', 'appointment_outcome', 'age_group']

for col in categorical_cols:
    print(health_connect_appointment[col].value_counts())
    print()

gender
Female               2488
Male                 2404
Prefer not to say     108
Name: count, dtype: int64

appointment_type
General Consultation       2086
Follow-up                  1421
Specialist Consultation     900
Diagnostic Test             593
Name: count, dtype: int64

reminder_sent
Yes    3634
No     1366
Name: count, dtype: int64

reminder_channel
SMS         2000
WhatsApp    1101
Email        533
Name: count, dtype: int64

appointment_day
Wednesday    737
Sunday       737
Friday       728
Saturday     724
Monday       701
Tuesday      689
Thursday     684
Name: count, dtype: int64

appointment_time
Morning      2227
Afternoon    2094
Evening       679
Name: count, dtype: int64

appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

age_group
65+      1241
35-44     819
55-64     800
45-54     793
25-34     783
18-24     564
Name: count, dtype: int64



In [11]:
health_connect_appointment['appointment_id'].duplicated().sum()

np.int64(0)

In [12]:
health_connect_appointment['patient_id'].nunique()

1696

In [13]:
health_connect_appointment['appointment_outcome'].value_counts()

appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

#### DATA INSPECTION SUMMARY
The HealthConnect appointment dataset contains 5,000 records across 18 variables, covering 1,696 unique patients.

Categorical variables were also checked for consistency. All categories were clean and as expected from the Data Dictionary, with one notable exception: gender contains three categories (Female, Male, Prefer not to say) rather than a simple binary split.

No duplicate appointment_id values were found, confirming the primary key is clean.

Three variables contain missing values which are
1. reminder_channel (1,366 missing)
2. distance_to_clinic_km (90 missing)
3. waiting_time_minutes (60 missing).

The missing data in reminder_channel is expected and logical because if reminder_sent is No it automatically means no value present for the reminder_channel. The gaps in distance and waiting time are minor and will need an imputation or exclusion decision before modelling.

This logic was also verified directly in code, all 1,366 missing reminder_channel values correspond exactly to reminder_sent = No, with no exceptions in either direction. Similarly, no records were found where previous_no_shows exceeds previous_appointments, confirming the historical counts are internally consistent.

The target column, appointment_outcome, splits into No-Show (2,423), Attended (2,314), and Cancelled (263). The two dominant classes are reasonably balanced, which is favourable for modelling, though the treatment of Cancelled needs a separate decision which will be addressed later

#### Problem Definition

HealthConnect Clinic loses appointment slots when patients miss their scheduled appointments. Currently, there is no way to identify which appointments are at risk of being missed. no-shows are only discovered after they happen, when the slot can no longer be reused.

This project treats the problem as a binary classification task: using information available about a patient and their appointment (such as appointment history, whether a reminder was sent, and distance to the clinic to predict whether the patient will Attend or No-Show.

A working prediction model would allow the clinic to identify high-risk appointments in advance and act early. for example, by sending extra reminders or confirming by phone instead of only discovering the loss on the day of the appointment.

Cancelled appointments (263 records) will be excluded from this prediction problem. A cancellation means the patient informed the clinic in advance, which is a different situation from a no-show, where there is no advance warning.

In [14]:
### Removing cancelled from the Target variable
health_connect_ml= health_connect_appointment[health_connect_appointment['appointment_outcome'] != 'Cancelled'].copy()
health_connect_ml['appointment_outcome'].value_counts()

appointment_outcome
No-Show     2423
Attended    2314
Name: count, dtype: int64

In [15]:
### Encoding Target Variable
health_connect_ml['appointment_outcome'] = health_connect_ml['appointment_outcome'].map({'Attended': 0, 'No-Show': 1})
health_connect_ml.head()

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,1
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,0
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,1
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,0
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,1


In [16]:
health_connect_ml.shape

(4737, 18)

#### Proposed Target Variable

The proposed target variable is appointment_outcome, restricted to the two outcomes relevant to this prediction problem, Attended and No-Show. As discussed in the Problem Definition, Cancelled appointments (263 records) are excluded, since a cancellation is a patient-initiated, known absence rather than an unpredicted no-show.

For modelling purposes, appointment_outcome is encoded numerically as

0 = Attended

1 = No-Show

After excluding Cancelled appointments, the dataset contains 4,737 records, split fairly evenly between No-Show (2,423) and Attended (2,314). This balance is favourable for modelling, as it reduces the risk of class imbalance issues that can arise when one outcome is much rarer than the other.

In [17]:
candidate_features = [
    'age', 'age_group', 'gender', 'appointment_type', 'booking_lead_days','previous_appointments', 'previous_no_shows', 'reminder_sent', 'reminder_channel', 'distance_to_clinic_km', 'waiting_time_minutes',
    'appointment_day', 'appointment_time']

health_connect_ml[candidate_features].dtypes

age                        int64
age_group                 object
gender                    object
appointment_type          object
booking_lead_days          int64
previous_appointments      int64
previous_no_shows          int64
reminder_sent             object
reminder_channel          object
distance_to_clinic_km    float64
waiting_time_minutes     float64
appointment_day           object
appointment_time          object
dtype: object

#### Potential Input Features

The following variables are considered potential input features for the no-show prediction model:

- **Age / Age Group:** Patient age may be associated with appointment attendance. The relationship between age and no-show behaviour will be investigated rather than assumed.
- **Gender:** Gender may show differences in appointment attendance patterns. This will be treated as a hypothesis to be tested, rather than an established relationship.
- **Appointment Type:** Different appointment types may have different attendance patterns. For example, follow-up appointments may potentially have different no-show rates, but this will need to be tested using the data.
- **Booking Lead Days:** The number of days between booking and the scheduled appointment may influence attendance, as longer waiting periods could potentially affect whether patients attend.
- **Previous Appointments:** The number of previous appointments may provide information about a patient's appointment history and could help identify attendance patterns.
- **Previous No-Shows:** Previous no-show behaviour may contain useful information about future appointment attendance. Its predictive value will be tested during subsequent modelling.
- **Reminder Sent:** Whether a reminder was sent may be associated with appointment attendance. However, this relationship should be interpreted as an association rather than evidence that reminders directly cause patients to attend. This variable should only be used if it is available at the time the prediction is made.
- **Reminder Channel:** The method used to send a reminder may potentially be associated with attendance. Its usefulness will be assessed during subsequent analysis and should only be included if the information is available at the prediction point.
- **Distance to Clinic:** Distance to the clinic may be associated with appointment attendance and is therefore considered a potential predictor.
- **Waiting Time:** Estimated waiting time may be associated with attendance. Its inclusion will depend on whether this information is available before the prediction is generated.
- **Appointment Day / Time:** Attendance patterns may vary across different days and appointment times. These variables will be explored to determine whether they provide useful predictive information.

These variables are considered **candidate features rather than confirmed predictors**. Their usefulness will be evaluated during subsequent analysis, while also considering data quality, availability at the prediction point, and potential data leakage.

#### Initial Modelling Approach

The proposed approach is to develop a binary classification model that predicts whether an appointment will result in **Attended** or **No-Show**, after excluding cancelled appointments.

**Baseline Model:** Logistic Regression will be considered as an initial baseline because it is relatively simple and interpretable. Other models, such as Random Forest, may be explored later if the data suggests that non-linear relationships or interactions are important.

**Prediction Timing:** The point at which the prediction will be generated needs to be clearly defined before the final feature set is selected. Only information that would genuinely be available at that point should be used. For example, `reminder_sent`, `reminder_channel`, and `waiting_time_minutes` should only be included if they are available before the prediction is made.

**Train-Test Split:** An initial 80/20 train-test split with stratification may be used to maintain the distribution of the target classes. However, the dataset contains multiple appointments for some patients. A random split could therefore place appointments belonging to the same patient in both the training and test sets, which may result in overly optimistic performance. A patient-level or group-based split should therefore be considered depending on the intended use of the model.

**Evaluation:** Accuracy will be considered alongside other classification metrics. Since the project aims to identify appointments at risk of no-show, **recall for the No-Show class**, as well as precision and F1-score, will be important evaluation metrics. The relative operational cost of false negatives and false positives should be confirmed with the clinic before the final evaluation criteria are established.

#### Key Modelling Considerations

**Missing Data:** `distance_to_clinic_km` and `waiting_time_minutes` contain missing values. Median imputation is proposed as an initial approach rather than removing affected rows. However, the distribution and missingness patterns should be examined before the final preprocessing approach is selected. Imputation values should be calculated using the training data only to avoid data leakage.

**Data Leakage:** Features should only be included if they are available at the time the prediction is generated. Variables that contain information generated after the appointment outcome occurs must not be used as predictors. The availability of reminder-related variables and estimated waiting time at the selected prediction point should therefore be confirmed.

**Repeated Patients:** The dataset contains 5,000 appointment records but only 1,696 unique patients. Since some patients therefore have multiple appointments, a random train-test split could result in appointments from the same patient appearing in both sets. A patient-level split should be considered to reduce the risk of overly optimistic model performance.

**Class Balance:** After excluding cancelled appointments, the two target classes are reasonably balanced, with 2,423 No-Show appointments and 2,314 Attended appointments. Severe class imbalance is therefore not an immediate concern, although class-specific performance should still be monitored during evaluation.

**Feature Interpretation:** Relationships identified between variables and appointment outcomes should be interpreted as associations unless a causal relationship can be established. In particular, variables such as reminders should not automatically be interpreted as evidence that a particular intervention causes better attendance.

**Ethical Considerations:** Demographic variables such as gender may be useful for understanding patterns in the data, but any observed differences should be investigated carefully. Statistical associations should not automatically result in different treatment of patients based solely on demographic characteristics.

#### Assumptions, Limitations, Risks and Dependencies

**Assumptions:**
- The definitions provided in the Data Dictionary accurately describe the variables and their intended use.
- The information required for prediction will be available at the selected prediction point.
- Cancelled appointments represent a separate outcome from no-shows and will therefore be excluded from the initial binary classification problem.

**Limitations:**
- The dataset is fictional and anonymised. Therefore, relationships identified in the dataset should not be assumed to generalise directly to a real healthcare setting without further validation.
- The dataset represents a fixed set of appointment records and may not capture changes in patient behaviour or clinic operations over time.
- The potential relationships identified between features and no-show behaviour are hypotheses that require further analysis and modelling.
- Multiple appointments may belong to the same patient, which creates an important consideration when splitting the data for model evaluation.

**Risks:**
- Including variables that are not available at the prediction point could result in data leakage and misleading model performance.
- Randomly splitting appointments from the same patients between training and test sets could produce overly optimistic performance estimates.
- Demographic variables could contribute to biased predictions if their relationships with appointment outcomes are not carefully evaluated.
- Observed associations, particularly around reminders, could be incorrectly interpreted as causal relationships.

**Dependencies:**
- Final feature selection depends on confirming which variables are available at the selected prediction point.
- Model development depends on the quality and definitions provided in the dataset and Data Dictionary.
- The final evaluation strategy depends on the intended population for prediction and whether the model needs to generalise to new patients or patients with previous appointment records.
- Further analysis in the next stage will be required to determine which candidate features provide meaningful predictive value.